In [1]:
# Environment setup & Imports
import os, sys
sys.path.append(os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML
from scipy.integrate import solve_ivp

from shared_utils.theme import PALETTE, apply_sleek_theme, create_dashboard_header
from nullcline_engine import compute_grid_derivatives
from linearization_classifier import classify_2d_jacobian
from competitive_lotka_volterra import (
    rabbits_vs_sheep, rabbits_vs_sheep_ode, compute_rabbits_sheep_equilibria, trace_separatrix
)

apply_sleek_theme()
display(HTML(create_dashboard_header(
    'ACT 03: THE GEOMETRY OF THE PHASE PLANE',
    'Interactive GUI Suite: Nullclines, Invariant Manifolds & Competitive Exclusion'
)))

In [2]:
# LAB 3.1: 2D LINEARIZATION & SPECTRAL MATRIX PLAYGROUND
display(HTML(create_dashboard_header(
    'LAB 3.1: JACOBIAN SPECTRAL CLASSIFIER & PHASE PORTRAIT',
    'Adjust 2x2 matrix coefficients to observe topological transitions: Nodes, Spirals, Saddles, Centers.'
)))

def render_linear_phase_portrait(a11, a12, a21, a22):
    J = np.array([[a11, a12], [a21, a22]])
    diag = classify_2d_jacobian(J)
    
    x_vals = np.linspace(-3.0, 3.0, 20)
    y_vals = np.linspace(-3.0, 3.0, 20)
    X, Y = np.meshgrid(x_vals, y_vals)
    U = a11 * X + a12 * Y
    V = a21 * X + a22 * Y
    
    fig, axs = plt.subplots(1, 2, figsize=(13, 5), dpi=100)
    
    # Panel A: Phase Plane Flow
    axs[0].streamplot(X, Y, U, V, color=PALETTE['border'], density=0.8, linewidth=0.7, arrowsize=0.8)
    seeds = [[2, 2], [-2, 2], [-2, -2], [2, -2], [1, -2], [-1, 2]]
    for s in seeds:
        sol = solve_ivp(lambda t, z: J @ z, (0, 6.0), s, t_eval=np.linspace(0, 6.0, 200))
        axs[0].plot(sol.y[0], sol.y[1], color=PALETTE['cyan'], lw=1.3)
    axs[0].scatter([0], [0], color=PALETTE['crimson'], s=80, zorder=6, label=str(diag['morphology']))
    axs[0].set_title('Flow Topography | ' + str(diag['stability']), fontsize=11)
    axs[0].set_xlabel('State x1', fontsize=10)
    axs[0].set_ylabel('State x2', fontsize=10)
    axs[0].set_xlim(-3, 3)
    axs[0].set_ylim(-3, 3)
    axs[0].legend(loc='upper right')
    
    # Panel B: Trace-Determinant Diagram
    tau_arr = np.linspace(-4, 4, 300)
    delta_parabola = 0.25 * (tau_arr ** 2)
    axs[1].plot(tau_arr, delta_parabola, color=PALETTE['amber'], lw=1.5, label='Parabola Delta = 1/4 tau^2')
    axs[1].axhline(0, color=PALETTE['border'], linestyle='--', lw=1.0)
    axs[1].axvline(0, color=PALETTE['border'], linestyle='--', lw=1.0)
    curr_label = 'System (tau={:.2f}, Delta={:.2f})'.format(diag['trace'], diag['det'])
    axs[1].scatter([diag['trace']], [diag['det']], color=PALETTE['crimson'], s=100, edgecolors=PALETTE['text_main'], lw=1.5, zorder=6, label=curr_label)
    axs[1].set_title('Trace-Determinant Plane', fontsize=11)
    axs[1].set_xlabel('Trace tau', fontsize=10)
    axs[1].set_ylabel('Determinant Delta', fontsize=10)
    axs[1].set_xlim(-4, 4)
    axs[1].set_ylim(-3, 5)
    axs[1].legend(loc='lower right')
    
    plt.tight_layout()
    plt.show()

s_a11 = widgets.FloatSlider(value=-0.5, min=-2.0, max=2.0, step=0.1, description='a11:', layout=widgets.Layout(width='320px'))
s_a12 = widgets.FloatSlider(value=1.5, min=-2.0, max=2.0, step=0.1, description='a12:', layout=widgets.Layout(width='320px'))
s_a21 = widgets.FloatSlider(value=-1.5, min=-2.0, max=2.0, step=0.1, description='a21:', layout=widgets.Layout(width='320px'))
s_a22 = widgets.FloatSlider(value=-0.5, min=-2.0, max=2.0, step=0.1, description='a22:', layout=widgets.Layout(width='320px'))

ui_lin = widgets.VBox([
    widgets.HBox([s_a11, s_a12]),
    widgets.HBox([s_a21, s_a22])
], layout=widgets.Layout(padding='10px', background_color=PALETTE['surface'], border='1px solid ' + PALETTE['border'], border_radius='8px'))

out_lin = widgets.interactive_output(render_linear_phase_portrait, {'a11': s_a11, 'a12': s_a12, 'a21': s_a21, 'a22': s_a22})
display(ui_lin, out_lin)

Output()

In [ ]:
# LAB 3.2: RABBITS VS. SHEEP COMPETITIVE EXCLUSION DASHBOARD
display(HTML(create_dashboard_header(
    'LAB 3.2: RABBITS VS. SHEEP, NULLCLINES & SEPARATRIX',
    'Strogatz competition model: Click sliders to inject initial populations and witness competitive exclusion.'
)))

def render_competition_dashboard(rabbit_init, sheep_init):
    X_grid, Y_grid, U, V, Un, Vn = compute_grid_derivatives(rabbits_vs_sheep, (0, 3.5), (0, 2.5), 25)
    eqs = compute_rabbits_sheep_equilibria()
    sep1, sep2 = trace_separatrix()
    
    sol_user = solve_ivp(rabbits_vs_sheep_ode, (0, 25.0), [rabbit_init, sheep_init], t_eval=np.linspace(0, 25.0, 500))
    
    fig, ax = plt.subplots(figsize=(10, 6.5), dpi=100)
    ax.streamplot(X_grid, Y_grid, U, V, color=PALETTE['border'], density=0.9, linewidth=0.6, arrowsize=0.8)
    
    x_line = np.linspace(0, 3.5, 200)
    ax.axvline(0, color=PALETTE['cyan'], linestyle='--', lw=1.2, label='x-nullcline')
    ax.plot(x_line, np.maximum(0, (3.0 - x_line) / 2.0), color=PALETTE['cyan'], linestyle='--', lw=1.5)
    
    ax.axhline(0, color=PALETTE['amber'], linestyle=':', lw=1.2, label='y-nullcline')
    ax.plot(x_line, np.maximum(0, 2.0 - x_line), color=PALETTE['amber'], linestyle=':', lw=1.5)
    
    ax.plot(sep1[0], sep1[1], color=PALETTE['crimson'], lw=2.2, label='Separatrix')
    ax.plot(sep2[0], sep2[1], color=PALETTE['crimson'], lw=2.2)
    
    ax.plot(sol_user.y[0], sol_user.y[1], color=PALETTE['emerald'], lw=2.5, label='User Trajectory')
    ax.scatter([rabbit_init], [sheep_init], color=PALETTE['emerald'], s=90, edgecolors=PALETTE['text_main'], zorder=7, label='Initial State')
    
    for eq in eqs:
        c = eq['coord']
        marker_color = PALETTE['crimson'] if 'Saddle' in eq['name'] else PALETTE['emerald']
        ax.scatter([c[0]], [c[1]], color=marker_color, s=80, edgecolors=PALETTE['text_main'], lw=1.2, zorder=6)
        ax.text(c[0] + 0.08, c[1] + 0.08, eq['name'], color=PALETTE['text_main'], fontsize=9, weight='bold')
        
    final_x, final_y = sol_user.y[0][-1], sol_user.y[1][-1]
    outcome = 'Rabbit Monopoly Wins (x=3, y=0)' if final_x > 2.0 else 'Sheep Monopoly Wins (x=0, y=2)'
    ax.set_title('Rabbits vs. Sheep Basin Battle | Outcome: ' + outcome, fontsize=12, pad=12)
    ax.set_xlabel('Rabbit Population x', fontsize=10)
    ax.set_ylabel('Sheep Population y', fontsize=10)
    ax.set_xlim(0, 3.5)
    ax.set_ylim(0, 2.5)
    ax.legend(loc='upper right', framealpha=0.9)
    plt.tight_layout()
    plt.show()

s_rabbit = widgets.FloatSlider(value=1.5, min=0.1, max=3.0, step=0.1, description='Rabbits x0:', layout=widgets.Layout(width='380px'))
s_sheep = widgets.FloatSlider(value=1.2, min=0.1, max=2.2, step=0.1, description='Sheep y0:', layout=widgets.Layout(width='380px'))
ui_comp = widgets.HBox([s_rabbit, s_sheep], layout=widgets.Layout(padding='10px', background_color=PALETTE['surface'], border='1px solid ' + PALETTE['border'], border_radius='8px'))
out_comp = widgets.interactive_output(render_competition_dashboard, {'rabbit_init': s_rabbit, 'sheep_init': s_sheep})
display(ui_comp, out_comp)

Output()